# Visualize structural metrics

Example of visualizing structural metrics such as distances, angles, and dihedral angles:

<video controls src="./assets/gluhut_cvs.webm">

Load a recording (unbinding of a sugar from GluHUT) as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("gluhut-unbinding.nanover.zip")

Set up the server with playback of the universe:

In [2]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: visualize structural metrics")
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [3]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Define a set of utility classes for extracting positional data, computing angles and distances, and representing them as lines and text:

In [4]:
import numpy.typing as npt
from nanover.trajectory import FrameData
from MDAnalysis.lib.distances import calc_dihedrals, calc_angles

def calculate_angle_label_position(a, b, c):
    # place a little ways inside the corner of the angle
    delta = (a+c)/2 - b
    dir = delta / np.linalg.norm(delta)
    return b + dir * .1

WHITE = [1.0, 1.0, 1.0, 1.0]
RED = [1.0, 0.0, 0.0, 1.0]
GREEN = [0.0, 1.0, 0.0, 1.0]
BLUE = [0.0, 0.5, 1.0, 1.0]

class PositionMetricVisual:
    color = WHITE

    def __init__(self, key: str, *args: npt.NDArray):
        self.key = key
        self.groups = args

    def compute_positions(self, frame: FrameData):
        return [np.mean(frame.particle_positions[atoms], axis=0) for atoms in self.groups]

    def render(self, frame: FrameData):
        pass

class DihedralObject(PositionMetricVisual):
    color = BLUE
    def render(self, frame: FrameData):
        a, b, c, d = self.compute_positions(frame)
        angle = np.degrees(calc_dihedrals(a, b, c, d))

        m = (b+c)/2
        utilities.objects.update_line(f"{self.key}.am", positions=[a, m], size=0.01, color=self.color)
        utilities.objects.update_line(f"{self.key}.dm", positions=[d, m], size=0.01, color=self.color)
        utilities.objects.update_line(f"{self.key}.bc", positions=[b, c], size=0.01, color=self.color)
        utilities.objects.update_label(self.key, position=calculate_angle_label_position(a, (b + c) / 2, d), text=f"{angle:.1f}deg", color=self.color)

class AngleObject(PositionMetricVisual):
    color = RED
    def render(self, frame: FrameData):
        a, b, c = self.compute_positions(frame)
        angle = np.degrees(calc_angles(a, b, c))

        utilities.objects.update_line(f"{self.key}.ab", positions=[a, b], size=0.01, color=self.color)
        utilities.objects.update_line(f"{self.key}.bc", positions=[b, c], size=0.01, color=self.color)
        utilities.objects.update_label(self.key, position=calculate_angle_label_position(a, b, c), text=f"{angle:.1f}deg", color=self.color)

class DistanceObject(PositionMetricVisual):
    color = GREEN
    def render(self, frame: FrameData):
        a, b = self.compute_positions(frame)
        distance = np.linalg.norm(b-a)

        utilities.objects.update_line(self.key, positions=[a, b], size=0.01, color=self.color)
        utilities.objects.update_label(self.key, position=(a+b)/2, text=f"{distance:.2f}nm", color=self.color)

Define the groups of atoms of interest and the structural metrics to visualize:

In [5]:
# rings
r1_atoms = [46, 47, 50, 55, 58, 79]  # top ring
r2_atoms = [ 2,  3,  4,  7, 24, 27]  # bottom ring
r3_atoms = [32, 33, 34, 35, 41, 40]  # exit p3

# ligand atoms
l1_atoms = [162]
l2_atoms = [173]
l3_atoms = [159]

metrics = [
    DihedralObject("D1", r1_atoms, l1_atoms, l2_atoms, l3_atoms),
    DihedralObject("D2", r2_atoms, r1_atoms, l1_atoms, l2_atoms),
    DihedralObject("D3", r3_atoms, r2_atoms, r1_atoms, l1_atoms),
    AngleObject("A1", r1_atoms, l1_atoms, l2_atoms),
    AngleObject("A2", r2_atoms, r1_atoms, l1_atoms),
    DistanceObject("D1", l1_atoms, r1_atoms),
]

Define and start a simple FrameListener that rerenders all the visuals when the frame is updated:

In [6]:
import numpy as np
from nanover.jupyter import FrameListener

class MetricVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects:
            for object in metrics:
                object.render(full_frame)

visuals = MetricVisuals.from_runner(imd_runner)
visuals.start()